<a href="https://colab.research.google.com/github/codeKrantz/AI-2010-FINAL-Project/blob/main/NCAA_Tournament_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NCAA March Madness AI 2010 Final Project
* Using data from Kaggle: https://www.kaggle.com/competitions/march-machine-learning-mania-2026/overview
* This notebook requires a KAGGLE_USERNAME and KAGGLE_KEY:


In [8]:
#@title Kaggle Username and API Key Validation
#this cell is used for crediential checking
from google.colab import userdata

u = userdata.get("KAGGLE_USERNAME")
k = userdata.get("KAGGLE_KEY")
print(repr(u))                       # should be your Kaggle username
print(len(k), k != k.strip())        # key length, and whether it has stray whitespace/newline

'conradkrantz'
39 True


In [9]:
#@title Kaggle Login
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [10]:
#@title Data Importing from Kaggle
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

import kagglehub
import pandas as pd

path = kagglehub.competition_download("march-machine-learning-mania-2026")

# See what's there
files = sorted(os.listdir(path))
print(files)

# Load the men's files (they start with "M"); swap to "W" for women's
men = {
    f.replace(".csv", ""): pd.read_csv(os.path.join(path, f), encoding="latin-1")
    for f in files
    if f.startswith("M") and f.endswith(".csv")
}

for name, df in men.items():
    print(name, df.shape)

100%|██████████| 34.8M/34.8M [00:00<00:00, 93.2MB/s]

Extracting files...


['Cities.csv', 'Conferences.csv', 'MConferenceTourneyGames.csv', 'MGameCities.csv', 'MMasseyOrdinals.csv', 'MNCAATourneyCompactResults.csv', 'MNCAATourneyDetailedResults.csv', 'MNCAATourneySeedRoundSlots.csv', 'MNCAATourneySeeds.csv', 'MNCAATourneySlots.csv', 'MRegularSeasonCompactResults.csv', 'MRegularSeasonDetailedResults.csv', 'MSeasons.csv', 'MSecondaryTourneyCompactResults.csv', 'MSecondaryTourneyTeams.csv', 'MTeamCoaches.csv', 'MTeamConferences.csv', 'MTeamSpellings.csv', 'MTeams.csv', 'SampleSubmissionStage1.csv', 'SampleSubmissionStage2.csv', 'WConferenceTourneyGames.csv', 'WGameCities.csv', 'WNCAATourneyCompactResults.csv', 'WNCAATourneyDetailedResults.csv', 'WNCAATourneySeeds.csv', 'WNCAATourneySlots.csv', 'WRegularSeasonCompactResults.csv', 'WRegularSeasonDetailedResults.csv', 'WSeasons.csv', 'WSecondaryTourneyCompactResults.csv', 'WSecondaryTourneyTeams.csv', 'WTeamConferences.csv', 'WTeamSpellings.csv', 'WTeams.csv']
MConferenceTourneyGames (7093, 5)
MGameCities (92438, 6

In [11]:
print(path)

/root/.cache/kagglehub/competitions/march-machine-learning-mania-2026


In [12]:
import pandas as pd
df = pd.read_csv(os.path.join(path, "MRegularSeasonDetailedResults.csv"))

In [13]:
df

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,10,1104,68,1328,62,N,0,27,58,...,10,16,22,10,22,8,18,9,2,20
1,2003,10,1272,70,1393,63,N,0,26,62,...,24,9,20,20,25,7,12,8,6,16
2,2003,11,1266,73,1437,61,N,0,24,58,...,26,14,23,31,22,9,12,2,5,23
3,2003,11,1296,56,1457,50,N,0,18,38,...,22,8,15,17,20,9,19,4,3,23
4,2003,11,1400,77,1208,71,N,0,30,61,...,16,17,27,21,15,12,10,7,1,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124524,2026,132,1335,88,1463,84,N,1,29,64,...,24,13,17,17,24,23,9,9,4,16
124525,2026,132,1345,80,1276,72,N,0,30,57,...,24,5,6,11,22,15,7,2,3,18
124526,2026,132,1378,70,1455,55,N,0,23,61,...,19,13,20,12,27,8,14,7,7,19
124527,2026,132,1433,70,1173,62,N,0,23,57,...,21,9,17,8,31,16,11,3,4,18


## Data Cleaning

In [14]:
#@title Load the data files used by the model
import os
import pandas as pd

# `path` is set by the Kaggle download cell above.
# Only the 4 files the models need are loaded here.
FILES = {
    "rs":      "MRegularSeasonDetailedResults.csv",  # features: regular-season box scores (2003+)
    "tourney": "MNCAATourneyDetailedResults.csv",    # labels: who won each NCAA tournament game
    "seeds":   "MNCAATourneySeeds.csv",              # seed-only baseline model
    "teams":   "MTeams.csv",                         # TeamID -> school name for readable output
}

missing = [f for f in FILES.values() if not os.path.exists(os.path.join(path, f))]
assert not missing, f"Missing files in {path}: {missing}"

def load(key):
    return pd.read_csv(os.path.join(path, FILES[key]), encoding="latin-1")

rs      = load("rs")
tourney = load("tourney")
seeds   = load("seeds")
teams   = load("teams")

# Summary of what was loaded
summary = pd.DataFrame([
    {
        "variable": name,
        "file": FILES[name],
        "rows": len(d),
        "columns": d.shape[1],
        "seasons": f"{d.Season.min()}-{d.Season.max()}" if "Season" in d else "-",
    }
    for name, d in {"rs": rs, "tourney": tourney, "seeds": seeds, "teams": teams}.items()
])
print(summary.to_string(index=False))

variable                              file   rows  columns   seasons
      rs MRegularSeasonDetailedResults.csv 124529       34 2003-2026
 tourney   MNCAATourneyDetailedResults.csv   1449       34 2003-2025
   seeds             MNCAATourneySeeds.csv   2694        3 1985-2026
   teams                        MTeams.csv    381        4         -
